# Case 44 预习：不用神经网络的 Q-learning

这是参考出版社 Catch 环境编写的有限状态预习，不是原始神经网络或金融 A2C 案例的复现。我们直接观察 `(fruit_row, fruit_column, basket_position)`，用 Q 表代替原例中的图像输入和神经网络。有限玩具环境的成功率不能外推为金融表现。

运行目录：本 Notebook 所在的 `warmup/`。只用 Python 标准库。


In [1]:
from pathlib import Path
from tabular_catch import step, target, run, check_transition_contract
print("transition:", step((0, 3, 2), 2))
print("terminal target:", target(-1, True, [100, 100, 100], 0.95))


transition: ((1, 3, 3), 0, False)
terminal target: -1


## Q 更新

`Q(s,a) ← Q(s,a) + α [r + γ max Q(s′,a′) − Q(s,a)]`；终止时目标只取 `r`。

动作 0/1/2 分别为左移／不动／右移。果实每步下降一行，篮子被环境截断在边界内。最后接到果实奖励 +1，否则 −1。边界约束是环境强制实现，不是奖励训练出来的。


In [2]:
old_q, alpha, gamma = 0.4, 0.25, 0.95
new_q = old_q + alpha * (target(0, False, [0.2, 0.8, 0.1], gamma) - old_q)
print("one Q update:", round(new_q, 4))
print("finite checks:", check_transition_contract())


one Q update: 0.49
finite checks: 1176


## 小实验：显式探索率

相同网格、学习率、折扣、训练次数；比较 epsilon=0 与 0.2，五个固定种子。训练中相同 Q 值随机打破平局，因此 epsilon=0 不意味着完全没有随机性。

评估固定策略，枚举原环境分布支持上的全部初始状态；与始终不动和随机动作比较。评估不更新 Q 表，但属于同一 MDP，不能称为未见市场数据上的泛化。


In [3]:
run(Path("results"))


Finite transition checks passed: 1176
tabular_q, epsilon=0.0: mean success=1.000, min=1.000, max=1.000
tabular_q, epsilon=0.2: mean success=1.000, min=1.000, max=1.000
stay, epsilon=: mean success=0.429, min=0.429, max=0.429
random, epsilon=: mean success=0.440, min=0.400, max=0.514


## 结果和讨论

完整指标见 `results/metrics.csv`，训练回报、评估轨迹和配置均一并保存。这里只描述本次有限环境结果，不能把两个探索率的表现解释为普遍优劣。

请自己解释：为什么这个玩具环境适合查表？为什么仅提高奖励不能证明约束成立？引入神经网络或金融数据后，哪些假设会改变？

下一步对照 `../upstream/7.1 Q-Learning.ipynb`，再进入金融 A2C 数据和代码。正式展示仍需完成老师分配案例的复现及扩展。
